# Golf Swing Data Pipeline & Feature Engineering Exploration

This notebook prototypes the parsing of the GolfDB dataset annotations (`golfDB.mat`), maps the 8 key swing events to relative frames in the local video files, runs the coordinate extraction pipeline, and engineers the **sliding window** temporal features for high-movement joints.

## Objectives:
1. Parse annotations from `data/golfDB.mat`.
2. Batch process golf swing videos using the Day 1 `GolfVideoProcessor`.
3. Apply a sliding window of $\pm 5$ frames for wrists, elbows, shoulders, and hips coordinates.
4. Label frames: exact milestone frames mapped to `1-8`, transitional frames to `0`.
5. Verify coordinate changes around the **Top of Backswing** milestone.
6. Create a visual display helper for Jupyter to inspect any row in the dataset.

In [ ]:
import os
import sys
import scipy.io
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

# Add project root to sys.path
PROJECT_ROOT = '/Users/sagar/Documents/ML/golf-analysis'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.data_processor import GolfVideoProcessor
from src.visualizer import draw_skeleton
from src.feature_engineer import engineer_sliding_window, HIGH_MOVEMENT_JOINTS

### Step 1: Parse the GolfDB Mat File

We load `data/golfDB.mat` using `scipy.io.loadmat` and construct a mapping of `video_id` to its 8 milestone indices. Since the `.mat` file contains absolute frame numbers relative to the original YouTube videos, we subtract the start frame (`events[0]`) to make them relative to our trimmed local videos.

In [ ]:
mat_path = os.path.join(PROJECT_ROOT, 'data/golfDB.mat')

def parse_golfdb_mat(mat_path):
    mat_data = scipy.io.loadmat(mat_path)
    db = mat_data['golfDB'][0]
    
    metadata = []
    for i in range(len(db)):
        rec = db[i]
        vid_id = int(rec['id'][0][0])
        events = rec['events'][0]
        
        # Calculate relative event indices
        start_frame = events[0]
        rel_events = events - start_frame
        milestones = rel_events[1:9]
        
        # Extract additional metadata fields
        player = str(rec['player'][0]) if len(rec['player']) > 0 else ''
        sex = str(rec['sex'][0]) if len(rec['sex']) > 0 else ''
        club = str(rec['club'][0]) if len(rec['club']) > 0 else ''
        view = str(rec['view'][0]) if len(rec['view']) > 0 else ''
        slow = int(rec['slow'][0][0]) if len(rec['slow']) > 0 else 0
        bbox = rec['bbox'][0].tolist() if len(rec['bbox']) > 0 else []
        
        metadata.append({
            'video_id': vid_id,
            'player': player,
            'sex': sex,
            'club': club,
            'view': view,
            'slow': slow,
            'start_frame': start_frame,
            'end_frame': events[9],
            'bbox': bbox,
            'milestones': milestones.tolist()
        })
        
    return pd.DataFrame(metadata)

df_meta = parse_golfdb_mat(mat_path)
print(f'Loaded metadata for {len(df_meta)} videos.')
df_meta.head()

### Step 2: Batch Process Video Landmarks

We loop through a subset of local videos (configured by `NUM_VIDEOS_TO_PROCESS`), process each video using `GolfVideoProcessor`, and stack the resulting DataFrames. This returns raw, smoothed, and normalized landmark coordinate streams.

In [ ]:
# NUM_VIDEOS_TO_PROCESS is configurable. Use 10 for testing/prototyping.
# Set to None (or len(df_meta)) to process the entire dataset.
NUM_VIDEOS_TO_PROCESS = 10
VIDEO_DIR = os.path.join(PROJECT_ROOT, 'data/videos_160/videos_160')

processed_dfs = []

print(f'Starting processing for first {NUM_VIDEOS_TO_PROCESS} videos...')
for i in range(NUM_VIDEOS_TO_PROCESS):
    video_id = df_meta.loc[i, 'video_id']
    video_path = os.path.join(VIDEO_DIR, f'{video_id}.mp4')
    
    if not os.path.exists(video_path):
        print(f'Warning: Video file not found: {video_path}')
        continue
        
    print(f'Processing video {video_id} ({i+1}/{NUM_VIDEOS_TO_PROCESS})...')
    try:
        processor = GolfVideoProcessor()
        df_vid = processor.process_video(video_path, video_id=video_id)
        processed_dfs.append(df_vid)
        processor.close()
    except Exception as e:
        print(f'Error processing video {video_id}: {e}')

# Concatenate all processed videos
df_raw_features = pd.concat(processed_dfs, ignore_index=True)
print(f'Combined features shape: {df_raw_features.shape}')

### Step 3: Engineer Sliding Window Features

For each frame $T$, we append the coordinates from $T-5$ and $T+5$ as new columns. We group by `video_id` during shift operations to prevent temporal bleed across different videos. 

Boundary frames ($T < 5$ or $T > N - 6$) are padded using backfill and forward-fill within their respective video groups to keep columns free of `NaN` values without losing milestone rows near boundaries.

In [ ]:
df_windowed = engineer_sliding_window(df_raw_features, HIGH_MOVEMENT_JOINTS)
print(f'Windowed features shape: {df_windowed.shape}')

### Step 4: Label Frames

We assign labels `1-8` to the exact milestone frames matching the metadata indices, and `0` to all other transitional frames.

In [ ]:
def assign_labels(df, df_meta):
    df_labeled = df.copy()
    
    # Create mapping of video_id to its milestone list
    meta_dict = df_meta.set_index('video_id')['milestones'].to_dict()
    
    # Initialize label column to 0
    df_labeled['label'] = 0
    
    # Iterate through groups and assign labels
    for vid_id, group in df_labeled.groupby('video_id'):
        if vid_id not in meta_dict:
            continue
        milestones = meta_dict[vid_id]
        
        # Assign labels 1-8 to the milestone frame indices
        for milestone_idx, frame_idx in enumerate(milestones):
            label_val = milestone_idx + 1
            
            # Match the exact frame in the video
            mask = (df_labeled['video_id'] == vid_id) & (df_labeled['frame_index'] == frame_idx)
            df_labeled.loc[mask, 'label'] = label_val
            
    # Merge video-level metadata columns into the frame-level DataFrame
    metadata_cols = ['video_id', 'player', 'sex', 'club', 'view', 'slow']
    df_labeled = df_labeled.merge(df_meta[metadata_cols], on='video_id', how='left')
            
    return df_labeled

df_final = assign_labels(df_windowed, df_meta)
print('Label distribution:')
print(df_final['label'].value_counts())

### Step 5: Save Master Dataset

In [ ]:
processed_dir = os.path.join(PROJECT_ROOT, 'data/processed')
os.makedirs(processed_dir, exist_ok=True)
master_csv_path = os.path.join(processed_dir, 'master_training_dataset.csv')

df_final.to_csv(master_csv_path, index=False)
print(f'Master training dataset saved to {master_csv_path}')

### Step 6: Visual & Biomechanical Verification

Let's check the wrist coordinates around the **Top of Backswing** (label 4).

**MediaPipe Coordinate System Notes**:
- Y increases downwards. Therefore, a higher joint position on-screen has a **smaller Y coordinate** value.
- At the Top of Backswing ($T$), the wrist is at its peak elevation (minimum Y coordinate value).
- In the backswing ($T-5$) and downswing ($T+5$), the wrist should be physically lower (larger Y coordinate values).

Let's print the actual values to verify this physical relationship.

In [ ]:
# Filter for Top of Backswing rows
top_rows = df_final[df_final['label'] == 4]

print('Verifying "Top of Backswing" (label = 4) wrist coordinates:\n')
for idx, row in top_rows.head(5).iterrows():
    vid_id = row['video_id']
    frame_idx = row['frame_index']
    
    y_t = row['norm_right_wrist_y']
    y_prev = row['norm_right_wrist_y_t-5']
    y_next = row['norm_right_wrist_y_t+5']
    
    is_valid = (y_prev > y_t) and (y_next > y_t)
    
    print(f'Video {vid_id}, Frame {frame_idx}:')
    print(f'  Right Wrist Y (T-5): {y_prev: .4f}')
    print(f'  Right Wrist Y (T):   {y_t: .4f}')
    print(f'  Right Wrist Y (T+5): {y_next: .4f}')
    print(f'  Wrist position lower at T-5 and T+5 (Y_t-5 > Y_t and Y_t+5 > Y_t)? {is_valid}\n')

### Step 7: Notebook Frame Display Function

Here we define `show_frame_with_milestone(row)` using our shared `draw_skeleton` visualization utility to view processed skeleton overlays directly in the notebook.

In [ ]:
def show_frame_with_milestone(row, video_dir=VIDEO_DIR):
    video_id = int(row['video_id'])
    frame_idx = int(row['frame_index'])
    label = int(row['label'])
    
    video_path = os.path.join(video_dir, f'{video_id}.mp4')
    
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print(f'Failed to read frame {frame_idx} from {video_path}')
        return
        
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Draw skeleton overlay using the imported shared draw_skeleton
    draw_skeleton(frame_rgb, row, prefix='smooth_', line_color=(0, 255, 0), joint_color=(255, 0, 0))
    
    event_names = {
        0: 'Transition',
        1: 'Address',
        2: 'Toe-up (Backswing)',
        3: 'Mid-backswing',
        4: 'Top of Backswing',
        5: 'Mid-downswing',
        6: 'Impact',
        7: 'Mid-follow-through',
        8: 'Finish'
    }
    
    label_text = f'Video {video_id} | Frame {frame_idx} - {event_names.get(label, "Unknown")}'
    
    # Draw text background and text
    cv2.rectangle(frame_rgb, (10, 10), (320, 45), (0, 0, 0), -1)
    cv2.putText(frame_rgb, label_text, (15, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    
    img = Image.fromarray(frame_rgb)
    display(img)

# Find a top of backswing frame and show it
top_row = df_final[df_final['label'] == 4].iloc[0]
show_frame_with_milestone(top_row)